In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
raw_path = project_root / 'data' / 'raw'
print('Project root:', project_root)
print('Raw data exists:', raw_path.exists())
for f in ['train.csv', 'store.csv', 'test.csv', 'sample_submission.csv']:
    p = raw_path / f
    print(f'{f}:', p.exists(), p.stat().st_size if p.exists() else 0)
print('---')
print('Dossiers créés:')
for folder in ['data/raw', 'data/processed', 'notebooks', 'src', 'reports']:
    p = project_root / folder
    print(folder, p.exists())


# Phase 1 - Inspection et qualité des données

Dans cette étape, on charge les jeux de données Rossmann, on vérifie leur structure commune, puis on contrôle les anomalies de qualité (types, dates, valeurs manquantes, jours fermés et couverture temporelle).

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
train = pd.read_csv(project_root / 'data' / 'raw' / 'train.csv')
store = pd.read_csv(project_root / 'data' / 'raw' / 'store.csv')

print('train shape:', train.shape)
print('store shape:', store.shape)
print('\ntrain columns:', list(train.columns))
print('store columns:', list(store.columns))
print('\ntrain head:\n', train.head().to_string(index=False))
print('\nstore head:\n', store.head().to_string(index=False))
print('\nJoin keys intersection:', train['Store'].nunique(), store['Store'].nunique())
print('Expected same store IDs:', set(train['Store'].unique()).issubset(set(store['Store'].unique())))


NameError: name 'project_root' is not defined

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
train = pd.read_csv(project_root / 'data' / 'raw' / 'train.csv')
store = pd.read_csv(project_root / 'data' / 'raw' / 'store.csv')

print('train dtypes:\n', train.dtypes)
print('\nstore dtypes:\n', store.dtypes)
print('\ntrain numeric summary:\n', train[['Sales','Customers','Open','Promo','SchoolHoliday']].describe().round(2).to_string())
print('\nstore numeric summary:\n', store[['CompetitionDistance','CompetitionOpenSinceMonth','CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear']].describe().round(2).to_string())
print('\nStateHoliday train unique:', sorted(train['StateHoliday'].astype(str).unique().tolist()))
print('StoreType unique:', sorted(store['StoreType'].astype(str).unique().tolist()))
print('Assortment unique:', sorted(store['Assortment'].astype(str).unique().tolist()))


In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
train = pd.read_csv(project_root / 'data' / 'raw' / 'train.csv')
train['Date'] = pd.to_datetime(train['Date'])
train['StateHoliday'] = train['StateHoliday'].replace({0: '0', '0': '0'})

print('Date min/max:', train['Date'].min(), train['Date'].max())
print('StateHoliday normalized unique:', sorted(train['StateHoliday'].astype(str).unique().tolist()))
print('Rows with non-standard state holiday:', train.loc[~train['StateHoliday'].isin(['0','a','b','c']), 'StateHoliday'].head().tolist())


In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
train = pd.read_csv(project_root / 'data' / 'raw' / 'train.csv')
store = pd.read_csv(project_root / 'data' / 'raw' / 'store.csv')
print('duplicated rows train:', train.duplicated().sum())
print('duplicated rows store:', store.duplicated().sum())
print('\nMissing train:\n', train.isna().sum().to_string())
print('\nMissing store:\n', store.isna().sum().to_string())
print('\nMissing ratio store:\n', (store.isna().mean() * 100).round(2).sort_values(ascending=False).to_string())


In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
train = pd.read_csv(project_root / 'data' / 'raw' / 'train.csv')
closed_days = train[train['Open'] == 0]
open_zero_sales = train[(train['Open'] == 1) & (train['Sales'] == 0)]

print('Open=0 rows:', len(closed_days))
print('Share of closed days:', round(len(closed_days) / len(train) * 100, 2), '%')
print('Open=1 and Sales=0 rows:', len(open_zero_sales))
print('Open=0 summary sales:', closed_days['Sales'].describe().to_string())
print('\nOpen=1 & Sales=0 sample:\n', open_zero_sales.head().to_string(index=False))


In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
train = pd.read_csv(project_root / 'data' / 'raw' / 'train.csv')
train['Date'] = pd.to_datetime(train['Date'])

store_dates = train.groupby('Store')['Date'].agg(['min', 'max', 'count'])
print('Stores with incomplete sequence:', int((store_dates['count'] < 365).sum()))
print('\nDate coverage summary:\n', store_dates.head().to_string())
calendar = pd.date_range(train['Date'].min(), train['Date'].max(), freq='D')
print('\nExpected days in range:', len(calendar))
print('Unique dates observed:', train['Date'].nunique())
missing_dates = set(calendar) - set(train['Date'])
print('Missing dates count:', len(missing_dates))
if len(missing_dates) > 0:
    print('First missing dates:', sorted(list(missing_dates))[:5])


In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
quality_summary = pd.DataFrame({
    'issue': [
        'Closed days',
        'Null sales when store open',
        'Missing CompetitionDistance',
        'Missing CompetitionOpenSince*',
        'Missing Promo2Since*',
        'Inconsistent StateHoliday encoding',
        'Potential calendar gaps by store'
    ],
    'description': [
        'Open=0 days are excluded from training and predicted with 0 sales.',
        'Open=1 and Sales=0 should be investigated as possible anomalies or genuine low-volume days.',
        'Impute with median + missing indicator.',
        'Do not blindly impute; encode as non-applicable / missing category.',
        'Do not blindly impute; encode as non-applicable / missing category.',
        'Normalize 0, 0.0, a, b, c to the same categorical style.',
        'Each store must keep a continuous date sequence for lag features.'
    ],
    'decision': [
        'Keep as separate rule for predictions.',
        'Audit and keep only if valid business context.',
        'Median + missing flag.',
        'Use sentinel value / category.',
        'Use sentinel value / category.',
        'Standardize before modeling.',
        'Check completeness before feature generation.'
    ]
})
print(quality_summary.to_string(index=False))
